# AutoGrow4 on SDSC Expanse

This notebook provides a complete environment for running **AutoGrow 4.0** on the SDSC Expanse supercomputer.  
It includes:
1. **Installation** of AutoGrow4 and all dependencies
2. **NGL Viewer** for receptor visualization and binding-box definition
3. **Interactive GUI** (ipywidgets) for configuring all AutoGrow4 parameters
4. **Slurm job builder** that generates and submits SBATCH scripts
5. **Job monitoring** to track progress and retrieve results

---

## 1. Installation & Setup

Run the cells below **once** to install AutoGrow4 and all required dependencies.  
These are designed for the SDSC Expanse environment (Linux, module system, conda).

In [ ]:
import os
import sys

# === Configuration ===
# Set your project directory (where everything will be installed)
PROJECT_DIR = os.path.expanduser("~/final_project")
AUTOGROW_DIR = os.path.join(PROJECT_DIR, "autogrow4")
MGLTOOLS_DIR = os.path.join(PROJECT_DIR, "mgltools_x86_64Linux2_1.5.7")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "autogrow_output")
RECEPTOR_DIR = os.path.join(PROJECT_DIR, "receptors")

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RECEPTOR_DIR, exist_ok=True)

# Default receptor: GID1 (Gibberellin Insensitive Dwarf 1)
DEFAULT_RECEPTOR = os.path.join(RECEPTOR_DIR, "2ZSH.pdb")

print(f"Project directory: {PROJECT_DIR}")
print(f"AutoGrow4 directory: {AUTOGROW_DIR}")
print(f"Receptor directory: {RECEPTOR_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

### 1.1 Install Python Dependencies (conda + pip)

In [ ]:
%%bash
# Install core Python dependencies via conda
conda install -y -c conda-forge rdkit numpy scipy matplotlib func_timeout 2>&1 | tail -5
echo "--- Core Python packages installed ---"

# Install mpi4py (required for supercomputer MPI runs)
conda install -y -c conda-forge mpi4py openmpi 2>&1 | tail -5
echo "--- mpi4py installed ---"

# Install nglview for molecular visualization
conda install -y -c conda-forge nglview 2>&1 | tail -5
echo "--- nglview installed ---"

# Install ipywidgets for the GUI
conda install -y -c conda-forge ipywidgets 2>&1 | tail -5
echo "--- ipywidgets installed ---"

# Install scoria for binding box calculations
pip install scoria 2>&1 | tail -3
echo "--- scoria installed ---"

echo "\n=== All Python dependencies installed ==="

### 1.2 Install OpenBabel

In [ ]:
%%bash
# Install OpenBabel via conda (preferred on Expanse — no sudo needed)
conda install -y -c conda-forge openbabel 2>&1 | tail -5

# Verify installation
echo "OpenBabel path: $(which obabel)"
obabel -V 2>&1 | head -1

### 1.3 Install MGLTools (Manual — NOT via pip/conda)

**Important:** MGLTools must be installed from the official tarball, NOT via pip or conda.  
This downloads the command-line version from Scripps.

In [ ]:
%%bash
cd ~/final_project

if [ ! -d "mgltools_x86_64Linux2_1.5.7" ]; then
    echo "Downloading MGLTools 1.5.6..."
    wget -q http://mgltools.scripps.edu/downloads/downloads/tars/releases/REL1.5.6/mgltools_x86_64Linux2_1.5.7.tar.gz
    
    echo "Extracting..."
    tar -xzf mgltools_x86_64Linux2_1.5.7.tar.gz
    
    echo "Installing MGLTools..."
    cd mgltools_x86_64Linux2_1.5.7
    
    # Check if MGLToolsPckgs needs to be extracted
    if [ ! -d "MGLToolsPckgs" ] && [ -f "MGLToolsPckgs.tar.gz" ]; then
        tar -xzf MGLToolsPckgs.tar.gz
    fi
    
    # Run install (non-interactive)
    echo "y" | bash install.sh 2>&1 | tail -5
    
    cd ..
    rm -f mgltools_x86_64Linux2_1.5.7.tar.gz
    echo "MGLTools installed successfully."
else
    echo "MGLTools already installed."
fi

# Verify key files exist
ls ~/final_project/mgltools_x86_64Linux2_1.5.7/bin/pythonsh 2>/dev/null && echo "✓ pythonsh found" || echo "✗ pythonsh NOT found"
ls ~/final_project/mgltools_x86_64Linux2_1.5.7/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_ligand4.py 2>/dev/null && echo "✓ prepare_ligand4.py found" || echo "✗ prepare_ligand4.py NOT found"

### 1.4 Clone AutoGrow4

In [ ]:
%%bash
cd ~/final_project

if [ ! -d "autogrow4" ]; then
    echo "Cloning AutoGrow4..."
    git clone https://github.com/durrantlab/autogrow4.git 2>&1 | tail -3
    echo "AutoGrow4 cloned successfully."
else
    echo "AutoGrow4 already present."
fi

# Verify
ls ~/final_project/autogrow4/run_autogrow.py && echo "✓ run_autogrow.py found"

### 1.4a Apply Compatibility Patches

AutoGrow4 requires patches for compatibility with **RDKit 2025+** and the `CompoundInfo` dataclass refactoring.

In [ ]:
%%bash
cd ~/final_project/autogrow4

echo "=== Applying AutoGrow4 compatibility patches ==="

# Patch 1: MyMol.py - RDKit 2025+ chirality parameter
MYMOL=autogrow/operators/convert_files/gypsum_dl/gypsum_dl/MyMol.py
python3 << 'PATCH1'
import re
path = "autogrow/operators/convert_files/gypsum_dl/gypsum_dl/MyMol.py"
content = open(path).read()
clean = "params.enforceChirality = True  # RDKit 2025+"
old_only = "params.enforcechiral = True"
if clean in content and content.count("try:") - content.count("except") == 0:
    print("  [SKIP] MyMol.py already cleanly patched")
else:
    pattern = r"(            # The default, but just a sanity check\.\n)(.*?)(            # Set a max number of times)"
    m = re.search(pattern, content, re.DOTALL)
    if m:
        repl = (m.group(1) +
            "            try:\n"
            "                params.enforceChirality = True  # RDKit 2025+\n"
            "            except AttributeError:\n"
            "                params.enforcechiral = True  # Older RDKit\n\n"
            + m.group(3))
        content = content[:m.start()] + repl + content[m.end():]
        open(path, "w").write(content)
        print("  [OK] MyMol.py: clean chirality patch")
    elif old_only in content:
        content = content.replace(
            "            params.enforcechiral = True",
            "            try:\n"
            "                params.enforceChirality = True  # RDKit 2025+\n"
            "            except AttributeError:\n"
            "                params.enforcechiral = True  # Older RDKit")
        open(path, "w").write(content)
        print("  [OK] MyMol.py: basic chirality patch")
    else:
        print("  [SKIP] MyMol.py: no enforcechiral found")
PATCH1

# Patch 2: types.py - CompoundInfo backward compat + score fix + append
TYPES=autogrow/types.py
python3 << 'PATCH2'
path = "autogrow/types.py"
content = open(path).read()
changed = False

if "__getitem__" not in content:
    new_methods = """
    def __getitem__(self, index):
        lst = self.to_list()
        return lst[index]

    def __len__(self):
        return len(self.to_list())

    def __bool__(self):
        return True

    def __iter__(self):
        return iter(self.to_list())

    def append(self, value):
        self.diversity_score = float(value)

"""
    content = content.replace(
        "    @property\n    def score",
        new_methods + "    @property\n    def score"
    )
    changed = True
    print("  [OK] types.py: added backward compat methods")

if "def append" not in content:
    content = content.replace(
        "    @property\n    def score",
        "    def append(self, value):\n"
        "        self.diversity_score = float(value)\n\n"
        "    @property\n    def score"
    )
    changed = True
    print("  [OK] types.py: added append method")

if "if self.docking_score:" in content:
    content = content.replace("if self.docking_score:", "if self.docking_score is not None:")
    content = content.replace("if self.diversity_score:", "if self.diversity_score is not None:")
    content = content.replace("if self.fitness_score:", "if self.fitness_score is not None:")
    changed = True
    print("  [OK] types.py: fixed score property truthiness")

if changed:
    open(path, "w").write(content)
else:
    print("  [SKIP] types.py: already patched")
PATCH2

# Patch 3: operations.py - CompoundInfo type checks
OPS=autogrow/operators/operations.py
if grep -q 'type(x) is list' "$OPS" 2>/dev/null; then
    sed -i "s/type(x) is list/not isinstance(x, str)/g" "$OPS"
    sed -i "s/type(x) == list/not isinstance(x, str)/g" "$OPS"
    echo "  [OK] operations.py: CompoundInfo type checks"
else
    echo "  [SKIP] operations.py: already patched"
fi

# Patch 4: NNScore placeholder files
mkdir -p autogrow/docking/scoring/nn_score_exe/nnscore1
mkdir -p autogrow/docking/scoring/nn_score_exe/nnscore2
[ ! -f autogrow/docking/scoring/nn_score_exe/nnscore1/NNScore.py ] && \
    echo '# NNScore1 placeholder' > autogrow/docking/scoring/nn_score_exe/nnscore1/NNScore.py
[ ! -f autogrow/docking/scoring/nn_score_exe/nnscore2/NNScore2.py ] && \
    echo '# NNScore2 placeholder' > autogrow/docking/scoring/nn_score_exe/nnscore2/NNScore2.py
echo "  [OK] NNScore placeholder files"

# Patch 5: prepare_ligand4.py - fix os.path.basename stripping input path
PREP_LIG=~/final_project/mgltools_x86_64Linux2_1.5.7/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_ligand4.py
if grep -q 'ligand_filename = os.path.basename(a)' "$PREP_LIG" 2>/dev/null; then
    sed -i 's/ligand_filename = os.path.basename(a)/ligand_filename = a/' "$PREP_LIG"
    echo "  [OK] prepare_ligand4.py: fixed path stripping"
else
    echo "  [SKIP] prepare_ligand4.py: already patched"
fi

# Patch 6: generate_line_plot.py - division by zero guard
python3 << 'PATCH6'
path = "autogrow/plotting/generate_line_plot.py"
content = open(path).read()
old = "gen_affinity_average = gen_affinity_sum / num_lines_counter"
new = "gen_affinity_average = gen_affinity_sum / num_lines_counter if num_lines_counter > 0 else 0.0"
if old in content and "if num_lines_counter > 0" not in content:
    content = content.replace(old, new, 1)
    open(path, "w").write(content)
    print("  [OK] generate_line_plot.py: division by zero guard")
else:
    print("  [SKIP] generate_line_plot.py: already patched")
PATCH6

# Patch 7: execute_scoring_mol.py - list to CompoundInfo conversion
python3 << 'PATCH7'
path = "autogrow/docking/scoring/execute_scoring_mol.py"
content = open(path).read()
if "isinstance(lig, list)" in content:
    print("  [SKIP] execute_scoring_mol.py: already patched")
else:
    old = """    lig_dict: Dict[str, CompoundInfo] = {}
    for lig in list_of_list_of_lig_data:
        if lig is None:
            continue
        lig_short_id = lig.short_id"""
    new = """    lig_dict: Dict[str, CompoundInfo] = {}
    for lig in list_of_list_of_lig_data:
        if lig is None:
            continue
        if isinstance(lig, list):
            if len(lig) >= 3:
                score_val = float(lig[-1]) if len(lig) > 3 else None
                lig = CompoundInfo(
                    smiles=str(lig[0]),
                    name=str(lig[1]),
                    short_id=str(lig[2]),
                    docking_score=score_val,
                    fitness_score=score_val,
                )
            else:
                continue
        lig_short_id = lig.short_id"""
    if old in content:
        content = content.replace(old, new, 1)
        open(path, "w").write(content)
        print("  [OK] execute_scoring_mol.py: list->CompoundInfo conversion")
    else:
        print("  [WARN] execute_scoring_mol.py: pattern not found")
PATCH7

echo ""
echo "All patches applied successfully!"


In [ ]:
# Fix nglview frontend version mismatch on SDSC Expanse
# JupyterLab may have nglview-js-widgets 3.1.5 but Python package declares 3.1.4
try:
    import nglview._frontend as _nf
    if _nf.__frontend_version__ == '3.1.4':
        _nf.__frontend_version__ = '3.1.5'
        print('Patched nglview frontend version: 3.1.4 -> 3.1.5')
    else:
        print(f'nglview frontend version: {_nf.__frontend_version__} (no patch needed)')
except Exception as e:
    print(f'nglview version check skipped: {e}')

### 1.4b Download GID1 Receptor (PDB 2ZSH & 3ED1)

AutoGrow4 will be tested with the **GID1 (Gibberellin Insensitive Dwarf 1)** receptor.

In [ ]:
%%bash
cd ~/final_project/receptors 2>/dev/null || mkdir -p ~/final_project/receptors && cd ~/final_project/receptors

# Download GID1 receptor structures from RCSB PDB
if [ ! -f "2ZSH.pdb" ]; then
    echo "Downloading 2ZSH (GID1 with GA3 + GID2)..."
    wget -q "https://files.rcsb.org/download/2ZSH.pdb" -O 2ZSH.pdb
    echo "  Downloaded: $(wc -l < 2ZSH.pdb) lines"
else
    echo "2ZSH.pdb already present."
fi

if [ ! -f "3ED1.pdb" ]; then
    echo "Downloading 3ED1 (GID1 with GA4)..."
    wget -q "https://files.rcsb.org/download/3ED1.pdb" -O 3ED1.pdb
    echo "  Downloaded: $(wc -l < 3ED1.pdb) lines"
else
    echo "3ED1.pdb already present."
fi

echo ""
echo "Receptor files:"
ls -la ~/final_project/receptors/*.pdb

# Convert PDB to PDBQT using MGLTools (required for AutoGrow4/Vina docking)
# Note: Do NOT use obabel for receptor conversion - AutoGrow4's MGLTools can't parse obabel PDBQT format
MGLTOOLS_BIN=~/final_project/mgltools_x86_64Linux2_1.5.7
for pdb in ~/final_project/receptors/*.pdb; do
    pdbqt="${pdb%.pdb}.pdbqt"
    if [ ! -f "$pdbqt" ]; then
        echo "Converting $(basename $pdb) to PDBQT via MGLTools..."
        $MGLTOOLS_BIN/bin/pythonsh $MGLTOOLS_BIN/MGLToolsPckgs/AutoDockTools/Utilities24/prepare_receptor4.py \
            -r "$pdb" -o "$pdbqt" -A hydrogens 2>&1
        echo "  Result: $(wc -l < "$pdbqt") lines"
    else
        echo "$(basename $pdbqt) already present."
    fi
done

### 1.5 Install Force Fields (CHARMM, AMBER, OpenFF)

These force fields are used for ligand optimization and binding-site analysis.

In [ ]:
%%bash
# Install OpenMM and force fields
conda install -y -c conda-forge openmm openmmforcefields 2>&1 | tail -5
echo "--- OpenMM + CHARMM/AMBER force fields (openmmforcefields) installed ---"

# Install OpenFF Toolkit + Sage force field
conda install -y -c conda-forge openff-toolkit openff-forcefields 2>&1 | tail -5
echo "--- OpenFF Toolkit + Sage force field installed ---"

# Install OpenFFCHARMM (ParmEd-based CHARMM compatibility)
pip install openmm-forcefields 2>&1 | tail -3
echo "--- OpenFF/CHARMM bridge installed ---"

echo "\n=== All force fields installed ==="

### 1.6 Verify Installation

In [ ]:
import importlib

deps = {
    "rdkit": "RDKit",
    "numpy": "NumPy",
    "scipy": "SciPy",
    "matplotlib": "Matplotlib",
    "func_timeout": "func_timeout",
    "mpi4py": "mpi4py",
    "nglview": "NGLView",
    "ipywidgets": "ipywidgets",
    "openmm": "OpenMM",
    "openmmforcefields": "openmmforcefields (CHARMM/AMBER)",
    "openff.toolkit": "OpenFF Toolkit",
}

print("Dependency Check:")
print("=" * 50)
all_ok = True
for module, name in deps.items():
    try:
        m = importlib.import_module(module)
        ver = getattr(m, "__version__", "OK")
        print(f"  [OK] {name}: {ver}")
    except ImportError:
        print(f"  [!!] {name}: NOT INSTALLED")
        all_ok = False

# Check obabel
import shutil
obabel = shutil.which("obabel")
print(f"  [{'OK' if obabel else '!!'}] OpenBabel: {obabel or 'NOT FOUND'}")

# Check MGLTools
mgl = os.path.exists(os.path.join(MGLTOOLS_DIR, "bin", "pythonsh"))
print(f"  [{'OK' if mgl else '!!'}] MGLTools: {MGLTOOLS_DIR if mgl else 'NOT FOUND'}")

# Check AutoGrow4
ag4 = os.path.exists(os.path.join(AUTOGROW_DIR, "run_autogrow.py"))
print(f"  [{'OK' if ag4 else '!!'}] AutoGrow4: {AUTOGROW_DIR if ag4 else 'NOT FOUND'}")

print("=" * 50)
if all_ok and obabel and mgl and ag4:
    print("All dependencies verified!")
else:
    print("Some dependencies are missing. Please check above.")

---

## 2. Receptor Visualization & Binding Box Definition

Use the **NGL Viewer** below to visualize your receptor and interactively define the docking box.  
**Red** (X), **Green** (Y), and **Blue** (Z) arrows show the box edges from the corner.  
A **black sphere** marks the box center. Adjust the sliders to position the box around the binding site.

The box center and size values will be used by AutoGrow4 for docking.

In [ ]:
import nglview as nv
import ipywidgets as widgets

# ══════════════════════════════════════════════════════════════════
# Load Receptor Structure
# ══════════════════════════════════════════════════════════════════

receptor_pdb = os.path.join(RECEPTOR_DIR, "2ZSH.pdb")

# Parse receptor with ProDy if available, otherwise use nglview directly
try:
    import prody
    receptor_prody = prody.parsePDB(receptor_pdb)
    view = nv.NGLWidget()
    view.add_component(receptor_prody.select('chain A'))
    print(f"Loaded receptor with ProDy: {receptor_pdb}")
except ImportError:
    view = nv.show_file(receptor_pdb)
    print(f"Loaded receptor with nglview (ProDy not available): {receptor_pdb}")

view.clear_representations()
view.add_representation("cartoon", opacity=0.5)
view.add_representation("licorice", selection="ligand", opacity=0.5)

# ══════════════════════════════════════════════════════════════════
# Interactive Binding Box Sliders
# ══════════════════════════════════════════════════════════════════
# Use these sliders to define the docking box center and size.
# Red (X), Green (Y), Blue (Z) arrows show the box edges.
# A black sphere marks the box center.
# The size vectors should encompass the binding site on the receptor.

# Initialize binding_box with defaults (updated by sliders)
binding_box = {
    "center_x": 51.0, "center_y": 59.5, "center_z": 37.4,
    "size_x": 17.0, "size_y": 18.0, "size_z": 18.0
}

# GID1 defaults: binding pocket near GA3 ligand
c_x = widgets.FloatSlider(value=51.0, min=0, max=100, step=0.5, description='X:')
c_y = widgets.FloatSlider(value=59.5, min=0, max=100, step=0.5, description='Y:')
c_z = widgets.FloatSlider(value=37.4, min=0, max=100, step=0.5, description='Z:')
s_x = widgets.FloatSlider(value=17.0, min=5, max=40, step=0.5, description='X:')
s_y = widgets.FloatSlider(value=18.0, min=5, max=40, step=0.5, description='Y:')
s_z = widgets.FloatSlider(value=18.0, min=5, max=40, step=0.5, description='Z:')

def update_box(cx, cy, cz, sx, sy, sz):
    if hasattr(view, "component_4"):
        view.remove_component(view.component_4)
    if hasattr(view, "component_3"):
        view.remove_component(view.component_3)
    if hasattr(view, "component_2"):
        view.remove_component(view.component_2)
    if hasattr(view, "component_1"):
        view.remove_component(view.component_1)
    start_point = [cx - sx/2, cy - sy/2, cz - sz/2]
    view.shape.add_arrow(start_point, [start_point[0] + sx, start_point[1], start_point[2]], [1, 0, 0], 0.5)
    view.shape.add_arrow(start_point, [start_point[0], start_point[1] + sy, start_point[2]], [0, 1, 0], 0.5)
    view.shape.add_arrow(start_point, [start_point[0], start_point[1], start_point[2] + sz], [0, 0, 1], 0.5)
    view.shape.add_sphere([cx, cy, cz], [0, 0, 0], 0.5)
    view.display()

    # Update global binding_box for config generation
    global binding_box
    binding_box = {
        "center_x": cx, "center_y": cy, "center_z": cz,
        "size_x": sx, "size_y": sy, "size_z": sz
    }

out = widgets.interactive_output(update_box, {
    'cx': c_x, 'cy': c_y, 'cz': c_z,
    'sx': s_x, 'sy': s_y, 'sz': s_z
})

center_ui = widgets.VBox([widgets.Label("Box Center"), c_x, c_y, c_z])
size_ui = widgets.VBox([widgets.Label("Box Size"), s_x, s_y, s_z])
ui = widgets.HBox([center_ui, size_ui])

widgets.VBox([view, ui])

---

## 3. AutoGrow4 Parameter Configuration (GUI)

Configure all AutoGrow4 parameters using the tabbed interface below.  
Changes are reflected in the Slurm submission script automatically.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import json
import os

# ══════════════════════════════════════════════════════════════
# TAB 1: Input Files & Paths
# ══════════════════════════════════════════════════════════════

w_receptor = widgets.Text(
    value=os.path.join(RECEPTOR_DIR, "2ZSH.pdbqt"),
    description="Receptor PDB:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "160px"}
)
w_source_compounds = widgets.Text(
    value=os.path.join(AUTOGROW_DIR, "source_compounds", "naphthalene_smiles.smi"),
    description="Source Compounds:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "160px"}
)
w_output_folder = widgets.Text(
    value=OUTPUT_DIR,
    description="Output Folder:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "160px"}
)
w_mgltools_dir = widgets.Text(
    value=MGLTOOLS_DIR,
    description="MGLTools Dir:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "160px"}
)
w_obabel_path = widgets.Text(
    value="",
    description="obabel path:",
    placeholder="Leave blank to use MGLTools instead",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "160px"}
)
w_conversion = widgets.Dropdown(
    options=["MGLToolsConversion", "ObabelConversion"],
    value="MGLToolsConversion",
    description="PDB Converter:",
    style={"description_width": "160px"}
)

tab_files = widgets.VBox([
    widgets.HTML("<h4>Input Files & Paths</h4>"),
    w_receptor, w_source_compounds, w_output_folder,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>PDB → PDBQT Conversion Tool</b>"),
    w_conversion, w_mgltools_dir, w_obabel_path,
])

# ══════════════════════════════════════════════════════════════
# TAB 2: Evolutionary Algorithm Settings
# ══════════════════════════════════════════════════════════════

w_num_gens = widgets.IntSlider(value=5, min=1, max=100, description="Generations:", style={"description_width": "160px"}, layout=widgets.Layout(width="70%"))

# First generation
w_mutants_first = widgets.IntText(value=50, description="Mutants (1st gen):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_crossovers_first = widgets.IntText(value=50, description="Crossovers (1st gen):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_elitism_first = widgets.IntText(value=10, description="Elitism (1st gen):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))

# Subsequent generations
w_mutants = widgets.IntText(value=50, description="Mutants (per gen):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_crossovers = widgets.IntText(value=50, description="Crossovers (per gen):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_elitism = widgets.IntText(value=50, description="Elitism (per gen):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))

# Seeding
w_top_seed = widgets.IntText(value=50, description="Top mols to seed:", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_diversity_seed = widgets.IntText(value=10, description="Diversity seed (1st):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_diversity_deprec = widgets.IntText(value=2, description="Diversity depreciation:", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))

# Selection
w_selector = widgets.Dropdown(
    options=["Rank_Selector", "Roulette_Selector", "Tournament_Selector"],
    value="Rank_Selector",
    description="Selector:",
    style={"description_width": "160px"}
)
w_tourn_size = widgets.FloatText(value=0.1, description="Tournament size:", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))

# Reaction library
w_rxn_library = widgets.Dropdown(
    options=["all_rxns", "click_chem_rxns", "robust_rxns", "Custom"],
    value="all_rxns",
    description="Reaction Library:",
    style={"description_width": "160px"}
)

# Options
w_start_new = widgets.Checkbox(value=True, description="Start a new run")
w_filter_source = widgets.Checkbox(value=True, description="Filter source compounds")
w_use_docked = widgets.Checkbox(value=True, description="Dock source compounds")
w_redock_elite = widgets.Checkbox(value=False, description="Redock elite from prev gen")
w_generate_plot = widgets.Checkbox(value=True, description="Generate fitness plot")
w_reduce_files = widgets.Checkbox(value=True, description="Reduce file sizes")

tab_evolution = widgets.VBox([
    widgets.HTML("<h4>Evolutionary Algorithm Settings</h4>"),
    w_num_gens,
    widgets.HTML("<b>First Generation</b>"),
    widgets.HBox([w_mutants_first, w_crossovers_first, w_elitism_first]),
    widgets.HTML("<b>Subsequent Generations</b>"),
    widgets.HBox([w_mutants, w_crossovers, w_elitism]),
    widgets.HTML("<hr><b>Seeding & Selection</b>"),
    widgets.HBox([w_top_seed, w_diversity_seed, w_diversity_deprec]),
    widgets.HBox([w_selector, w_tourn_size]),
    widgets.HTML("<hr><b>Reaction Library</b>"),
    w_rxn_library,
    widgets.HTML("<hr><b>Options</b>"),
    widgets.HBox([w_start_new, w_filter_source, w_use_docked]),
    widgets.HBox([w_redock_elite, w_generate_plot, w_reduce_files]),
])

# ══════════════════════════════════════════════════════════════
# TAB 3: Filters
# ══════════════════════════════════════════════════════════════

w_lipinski_strict = widgets.Checkbox(value=False, description="Lipinski Strict")
w_lipinski_lenient = widgets.Checkbox(value=True, description="Lipinski Lenient")
w_ghose = widgets.Checkbox(value=False, description="Ghose")
w_ghose_modified = widgets.Checkbox(value=False, description="Ghose Modified")
w_mozziconacci = widgets.Checkbox(value=False, description="Mozziconacci")
w_vandewaterbeemd = widgets.Checkbox(value=False, description="Van de Waterbeemd")
w_pains = widgets.Checkbox(value=False, description="PAINS")
w_nih = widgets.Checkbox(value=False, description="NIH")
w_brenk = widgets.Checkbox(value=False, description="BRENK")
w_no_filters = widgets.Checkbox(value=False, description="No Filters (disable all)")

filter_info = widgets.HTML("""
<style>table.finfo td { padding: 2px 10px; font-size: 12px; }</style>
<table class="finfo">
<tr><td><b>Lipinski Strict</b></td><td>MW ≤ 500, LogP ≤ 5, H-donors ≤ 5, H-acceptors ≤ 10</td></tr>
<tr><td><b>Lipinski Lenient</b></td><td>Passes if meets 3 of 4 Lipinski criteria</td></tr>
<tr><td><b>Ghose</b></td><td>MW 160-480, LogP -0.4-5.6, atoms 20-70, refractivity 40-130</td></tr>
<tr><td><b>PAINS</b></td><td>Pan-assay interference compounds filter</td></tr>
<tr><td><b>NIH</b></td><td>NIH chemical genomics center substructure filter</td></tr>
<tr><td><b>BRENK</b></td><td>Brenk unwanted substructure filter</td></tr>
</table>
""")

tab_filters = widgets.VBox([
    widgets.HTML("<h4>Ligand Filters</h4>"),
    widgets.HTML("<p>Select which drug-likeness filters to apply to generated molecules:</p>"),
    widgets.HBox([
        widgets.VBox([w_lipinski_strict, w_lipinski_lenient, w_ghose, w_ghose_modified, w_mozziconacci]),
        widgets.VBox([w_vandewaterbeemd, w_pains, w_nih, w_brenk, w_no_filters]),
    ]),
    widgets.HTML("<hr>"),
    filter_info,
])

# ══════════════════════════════════════════════════════════════
# TAB 4: Docking & Scoring
# ══════════════════════════════════════════════════════════════

w_dock_choice = widgets.Dropdown(
    options=["VinaDocking", "QuickVina2Docking"],
    value="QuickVina2Docking",
    description="Docking Engine:",
    style={"description_width": "160px"}
)
w_exhaustiveness = widgets.IntSlider(value=8, min=1, max=32, description="Exhaustiveness:", style={"description_width": "160px"}, layout=widgets.Layout(width="60%"))
w_num_modes = widgets.IntSlider(value=9, min=1, max=20, description="Num Modes:", style={"description_width": "160px"}, layout=widgets.Layout(width="60%"))
w_dock_timeout = widgets.IntText(value=120, description="Dock Timeout (s):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))

w_scoring = widgets.Dropdown(
    options=["VINA", "NN1", "NN2"],
    value="VINA",
    description="Scoring Function:",
    style={"description_width": "160px"}
)
w_lig_efficiency = widgets.Checkbox(value=True, description="Rescore with ligand efficiency")

scoring_info = widgets.HTML("""
<style>table.sinfo td { padding: 2px 10px; font-size: 12px; }</style>
<table class="sinfo">
<tr><td><b>VINA</b></td><td>Uses Autodock Vina's built-in scoring (primary docking score)</td></tr>
<tr><td><b>NN1 (NNScore1)</b></td><td>Neural-network rescoring (trained on Vina 1.1.2 outputs)</td></tr>
<tr><td><b>NN2 (NNScore2)</b></td><td>Updated neural-network rescoring with improved predictions</td></tr>
<tr><td><b>Ligand Efficiency</b></td><td>Score / number of heavy atoms (normalizes for molecule size)</td></tr>
</table>
<p><i>Note: NNScore1 and NNScore2 require Autodock Vina 1.1.2 as the docking engine.</i></p>
""")

tab_docking = widgets.VBox([
    widgets.HTML("<h4>Docking & Scoring</h4>"),
    w_dock_choice, w_exhaustiveness, w_num_modes, w_dock_timeout,
    widgets.HTML("<hr><b>Scoring / Rescoring</b>"),
    widgets.HBox([w_scoring, w_lig_efficiency]),
    widgets.HTML("<hr>"),
    scoring_info,
])

# ══════════════════════════════════════════════════════════════
# TAB 5: Force Fields
# ══════════════════════════════════════════════════════════════

w_ff_choice = widgets.SelectMultiple(
    options=["CHARMM (openmmforcefields)", "AMBER (openmmforcefields)", "OpenFF Sage 2.0", "SQM2.20 (Semi-empirical)"],
    value=["CHARMM (openmmforcefields)", "AMBER (openmmforcefields)"],
    description="Force Fields:",
    rows=4,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="60%")
)

ff_info = widgets.HTML("""
<style>table.ffinfo td { padding: 2px 10px; font-size: 12px; }</style>
<table class="ffinfo">
<tr><td><b>CHARMM</b></td><td>Chemistry at HARvard Macromolecular Mechanics — general-purpose biomolecular FF</td></tr>
<tr><td><b>AMBER</b></td><td>Assisted Model Building with Energy Refinement — widely used for proteins/nucleic acids</td></tr>
<tr><td><b>OpenFF Sage</b></td><td>Open Force Field Initiative Sage 2.0 — modern, data-driven small-molecule FF</td></tr>
<tr><td><b>SQM2.20</b></td><td>Semi-empirical quantum mechanics method for energy refinement</td></tr>
</table>
<p><i>Force fields are available via openmmforcefields and openff-toolkit (installed above).<br>
They can be used for post-docking energy minimization and ligand optimization.</i></p>
""")

tab_forcefields = widgets.VBox([
    widgets.HTML("<h4>Force Fields</h4>"),
    widgets.HTML("<p>Select force fields available for ligand optimization (Ctrl+Click for multiple):</p>"),
    w_ff_choice,
    widgets.HTML("<hr>"),
    ff_info,
])

# ══════════════════════════════════════════════════════════════
# TAB 6: Gypsum-DL (3D Conversion)
# ══════════════════════════════════════════════════════════════

w_max_variants = widgets.IntSlider(value=3, min=1, max=10, description="Max variants/mol:", style={"description_width": "160px"}, layout=widgets.Layout(width="60%"))
w_gypsum_thoroughness = widgets.IntSlider(value=3, min=1, max=10, description="Thoroughness:", style={"description_width": "160px"}, layout=widgets.Layout(width="60%"))
w_gypsum_timeout = widgets.IntText(value=15, description="Timeout (min):", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_min_ph = widgets.FloatText(value=6.4, description="Min pH:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_max_ph = widgets.FloatText(value=8.4, description="Max pH:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_pka_precision = widgets.FloatText(value=1.0, description="pKa precision:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))

tab_gypsum = widgets.VBox([
    widgets.HTML("<h4>Gypsum-DL Settings (SMILES → 3D Conversion)</h4>"),
    w_max_variants, w_gypsum_thoroughness, w_gypsum_timeout,
    widgets.HTML("<hr><b>pH / Ionization</b>"),
    widgets.HBox([w_min_ph, w_max_ph, w_pka_precision]),
])

# ══════════════════════════════════════════════════════════════
# TAB 7: Slurm / HPC Settings
# ══════════════════════════════════════════════════════════════

w_slurm_account = widgets.Text(value="iit130", description="Account:", style={"description_width": "160px"}, layout=widgets.Layout(width="400px"))
w_slurm_partition = widgets.Dropdown(
    options=["shared", "compute", "gpu", "gpu-shared", "large-shared", "debug"],
    value="shared",
    description="Partition:",
    style={"description_width": "160px"}
)
w_slurm_nodes = widgets.IntText(value=1, description="Nodes:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_slurm_ntasks = widgets.IntText(value=4, description="Tasks/node:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_slurm_mem = widgets.Text(value="10G", description="Memory:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_slurm_time = widgets.Text(value="24:00:00", description="Wall time:", style={"description_width": "160px"}, layout=widgets.Layout(width="250px"))
w_slurm_jobname = widgets.Text(value="autogrow4", description="Job name:", style={"description_width": "160px"}, layout=widgets.Layout(width="400px"))
w_slurm_email = widgets.Text(value="", description="Email (optional):", placeholder="your@email.edu", style={"description_width": "160px"}, layout=widgets.Layout(width="400px"))
w_num_procs = widgets.IntText(value=-1, description="Num processors:", style={"description_width": "160px"}, layout=widgets.Layout(width="300px"))
w_multithread = widgets.Dropdown(
    options=["multithreading", "mpi", "serial"],
    value="multithreading",
    description="Parallel mode:",
    style={"description_width": "160px"}
)

tab_slurm = widgets.VBox([
    widgets.HTML("<h4>Slurm / HPC Settings</h4>"),
    widgets.HBox([w_slurm_account, w_slurm_partition]),
    widgets.HBox([w_slurm_nodes, w_slurm_ntasks]),
    widgets.HBox([w_slurm_mem, w_slurm_time]),
    w_slurm_jobname, w_slurm_email,
    widgets.HTML("<hr><b>Parallelization</b>"),
    widgets.HBox([w_num_procs, w_multithread]),
])

# ══════════════════════════════════════════════════════════════
# Assemble Tabs
# ══════════════════════════════════════════════════════════════

tabs = widgets.Tab()
tabs.children = [tab_files, tab_evolution, tab_filters, tab_docking, tab_forcefields, tab_gypsum, tab_slurm]
for i, title in enumerate(["Files/Paths", "Evolution", "Filters", "Docking/Scoring", "Force Fields", "Gypsum-DL", "Slurm/HPC"]):
    tabs.set_title(i, title)

display(tabs)

---

## 4. Generate Configuration & Slurm Script

Click **"Generate & Preview"** to build the AutoGrow4 JSON config and Slurm SBATCH script from the GUI settings above.

In [ ]:
import json
import os
from IPython.display import display, HTML
import ipywidgets as widgets

preview_output = widgets.Output()
generate_btn = widgets.Button(description="Generate & Preview", button_style="primary", layout=widgets.Layout(width="250px"))

def build_autogrow_config():
    """Build AutoGrow4 JSON config from widget values."""
    config = {
        # Receptor & binding box
        "filename_of_receptor": w_receptor.value,
        "center_x": binding_box["center_x"],
        "center_y": binding_box["center_y"],
        "center_z": binding_box["center_z"],
        "size_x": binding_box["size_x"],
        "size_y": binding_box["size_y"],
        "size_z": binding_box["size_z"],

        # Files
        "source_compound_file": w_source_compounds.value,
        "root_output_folder": w_output_folder.value,

        # Conversion
        "conversion_choice": w_conversion.value,
        "mgltools_directory": w_mgltools_dir.value if w_conversion.value == "MGLToolsConversion" else "",
        "obabel_path": w_obabel_path.value if w_conversion.value == "ObabelConversion" else "",

        # Evolution
        "num_generations": w_num_gens.value,
        "number_of_mutants_first_generation": w_mutants_first.value,
        "number_of_crossovers_first_generation": w_crossovers_first.value,
        "number_elitism_advance_from_previous_gen_first_generation": w_elitism_first.value,
        "number_of_mutants": w_mutants.value,
        "number_of_crossovers": w_crossovers.value,
        "number_elitism_advance_from_previous_gen": w_elitism.value,
        "top_mols_to_seed_next_generation": w_top_seed.value,
        "diversity_mols_to_seed_first_generation": w_diversity_seed.value,
        "diversity_seed_depreciation_per_gen": w_diversity_deprec.value,

        # Selection
        "selector_choice": w_selector.value,
        "tourn_size": w_tourn_size.value,

        # Reaction library
        "rxn_library": w_rxn_library.value,

        # Options
        "start_a_new_run": w_start_new.value,
        "filter_source_compounds": w_filter_source.value,
        "use_docked_source_compounds": w_use_docked.value,
        "redock_elite_from_previous_gen": w_redock_elite.value,
        "generate_plot": w_generate_plot.value,
        "reduce_files_sizes": w_reduce_files.value,

        # Docking
        "dock_choice": w_dock_choice.value,
        "docking_exhaustiveness": w_exhaustiveness.value,
        "docking_num_modes": w_num_modes.value,
        "docking_timeout_limit": w_dock_timeout.value,

        # Scoring
        "scoring_choice": w_scoring.value,
        "rescore_lig_efficiency": w_lig_efficiency.value,

        # Gypsum-DL
        "max_variants_per_compound": w_max_variants.value,
        "gypsum_thoroughness": w_gypsum_thoroughness.value,
        "gypsum_timeout_limit": w_gypsum_timeout.value,
        "min_ph": w_min_ph.value,
        "max_ph": w_max_ph.value,
        "pka_precision": w_pka_precision.value,

        # Parallelization
        "number_of_processors": w_num_procs.value,
        "multithread_mode": w_multithread.value,
        "mgl_python": "",
        "prepare_ligand4.py": "",
        "prepare_receptor4.py": "",
        "custom_conversion_script": "",
        "timeout_limit": 120,
        "cache_prerun": False,
    }

    # Filters
    if w_no_filters.value:
        config["No_Filters"] = True
    else:
        filter_map = {
            "LipinskiStrictFilter": w_lipinski_strict.value,
            "LipinskiLenientFilter": w_lipinski_lenient.value,
            "GhoseFilter": w_ghose.value,
            "GhoseModifiedFilter": w_ghose_modified.value,
            "MozziconacciFilter": w_mozziconacci.value,
            "VandeWaterbeemdFilter": w_vandewaterbeemd.value,
            "PAINSFilter": w_pains.value,
            "NIHFilter": w_nih.value,
            "BRENKFilter": w_brenk.value,
        }
        for k, v in filter_map.items():
            if v:
                config[k] = True

    # Remove empty strings, but keep keys AutoGrow4 expects
    required_keys = {"mgl_python", "prepare_ligand4.py", "prepare_receptor4.py", "custom_conversion_script"}
    config = {k: v for k, v in config.items() if v != "" or k in required_keys}

    return config


def build_slurm_script(config, json_path):
    """Build SBATCH script for Slurm submission."""
    lines = ["#!/bin/bash"]
    lines.append(f"#SBATCH --account={w_slurm_account.value}")
    lines.append(f"#SBATCH --partition={w_slurm_partition.value}")
    lines.append(f"#SBATCH --nodes={w_slurm_nodes.value}")
    lines.append(f"#SBATCH --ntasks-per-node={w_slurm_ntasks.value}")
    lines.append(f"#SBATCH --mem={w_slurm_mem.value}")
    lines.append(f"#SBATCH --time={w_slurm_time.value}")
    lines.append(f"#SBATCH --job-name=\"{w_slurm_jobname.value}\"")
    lines.append(f"#SBATCH --output=\"{w_slurm_jobname.value}.%j.%N.out\"")
    lines.append("#SBATCH --export=ALL")

    if w_slurm_email.value.strip():
        lines.append(f"#SBATCH --mail-user={w_slurm_email.value.strip()}")
        lines.append("#SBATCH --mail-type=ALL")

    lines.append("")
    lines.append("# ── Environment setup ──")
    lines.append("module purge")
    lines.append("module load cpu")
    lines.append("")
    lines.append("# Activate conda environment with all AutoGrow4 dependencies")
    lines.append("source ~/.bashrc")
    lines.append("conda activate ag4_full")
    lines.append("")
    lines.append(f"cd {AUTOGROW_DIR}")
    lines.append("")

    # Cache prerun first (prevents race conditions)
    lines.append("# Step 1: Cache prerun (single processor, prevents race conditions)")
    if w_multithread.value == "mpi":
        lines.append(f"srun -n 1 python run_autogrow.py -c")
        lines.append("")
        lines.append("# Step 2: Run AutoGrow4 with MPI")
        ntasks_total = w_slurm_nodes.value * w_slurm_ntasks.value
        lines.append(f"mpirun -n {ntasks_total} python -m mpi4py run_autogrow.py -j {json_path}")
    else:
        lines.append(f"srun -n 1 python run_autogrow.py -c")
        lines.append("")
        lines.append("# Step 2: Run AutoGrow4 with multiprocessing")
        lines.append(f"python run_autogrow.py -j {json_path}")

    lines.append("")
    lines.append("echo 'AutoGrow4 run completed.'")

    return "\n".join(lines)


def on_generate(btn):
    preview_output.clear_output()
    with preview_output:
        config = build_autogrow_config()
        json_path = os.path.join(PROJECT_DIR, "autogrow_config.json")
        slurm_path = os.path.join(PROJECT_DIR, "autogrow_slurm.sh")

        # Save JSON config
        with open(json_path, "w") as f:
            json.dump(config, f, indent=4)

        # Build and save Slurm script
        slurm_script = build_slurm_script(config, json_path)
        with open(slurm_path, "w") as f:
            f.write(slurm_script)

        display(HTML(f"<h4>AutoGrow4 Config: <code>{json_path}</code></h4>"))
        print(json.dumps(config, indent=2))

        display(HTML(f"<hr><h4>Slurm Script: <code>{slurm_path}</code></h4>"))
        print(slurm_script)

        display(HTML("<hr><p style='color:green'><b>Files saved! Ready to submit.</b></p>"))

generate_btn.on_click(on_generate)

display(generate_btn)
display(preview_output)

---

## 5. Submit Job to Expanse

Click **"Submit Job"** to launch the AutoGrow4 run on SDSC Expanse via Slurm.

In [ ]:
import subprocess
import ipywidgets as widgets
from IPython.display import display, HTML

submit_output = widgets.Output()
submit_btn = widgets.Button(description="Submit Job", button_style="success", layout=widgets.Layout(width="200px"))

slurm_job_id = [None]  # Track for monitoring

def on_submit(btn):
    submit_output.clear_output()
    with submit_output:
        slurm_path = os.path.join(PROJECT_DIR, "autogrow_slurm.sh")
        json_path = os.path.join(PROJECT_DIR, "autogrow_config.json")

        if not os.path.exists(slurm_path):
            display(HTML("<span style='color:red'>No Slurm script found. Click 'Generate & Preview' first.</span>"))
            return

        if not os.path.exists(json_path):
            display(HTML("<span style='color:red'>No config JSON found. Click 'Generate & Preview' first.</span>"))
            return

        print("Submitting job to Slurm...")
        try:
            result = subprocess.run(
                ["sbatch", slurm_path],
                capture_output=True, text=True, cwd=PROJECT_DIR
            )
            if result.returncode == 0:
                output = result.stdout.strip()
                print(f"Slurm response: {output}")
                # Extract job ID
                parts = output.split()
                if len(parts) >= 4:
                    slurm_job_id[0] = parts[-1]
                    display(HTML(f"<p style='color:green'><b>Job submitted successfully! Job ID: {slurm_job_id[0]}</b></p>"))
                else:
                    display(HTML(f"<p style='color:green'><b>Job submitted!</b> {output}</p>"))
            else:
                print(f"STDERR: {result.stderr}")
                display(HTML(f"<span style='color:red'>Submission failed: {result.stderr}</span>"))
        except FileNotFoundError:
            display(HTML("<span style='color:red'>sbatch not found. Are you on a Slurm-enabled node?</span>"))
        except Exception as e:
            display(HTML(f"<span style='color:red'>Error: {e}</span>"))

submit_btn.on_click(on_submit)

display(submit_btn)
display(submit_output)

---

## 6. Monitor Job & Retrieve Results

Use the cells below to check job status, stream logs, and view results.

In [ ]:
import subprocess
import glob
import ipywidgets as widgets
from IPython.display import display, HTML

monitor_output = widgets.Output()
check_btn = widgets.Button(description="Check Job Status", button_style="info", layout=widgets.Layout(width="200px"))
logs_btn = widgets.Button(description="View Latest Logs", button_style="warning", layout=widgets.Layout(width="200px"))
cancel_btn = widgets.Button(description="Cancel Job", button_style="danger", layout=widgets.Layout(width="200px"))

def on_check(btn):
    monitor_output.clear_output()
    with monitor_output:
        try:
            # Show queue status
            result = subprocess.run(["squeue", "-u", os.environ.get("USER", "$USER"), "--format=%.18i %.9P %.30j %.8u %.8T %.10M %.9l %.6D %R"],
                                   capture_output=True, text=True)
            if result.stdout.strip():
                print("=== Your Active Jobs ===")
                print(result.stdout)
            else:
                print("No active jobs found in the queue.")

            # Check specific job if we have an ID
            if slurm_job_id[0]:
                result2 = subprocess.run(["sacct", "-j", slurm_job_id[0], "--format=JobID,State,ExitCode,Elapsed,MaxRSS"],
                                        capture_output=True, text=True)
                if result2.stdout.strip():
                    print(f"\n=== Job {slurm_job_id[0]} Details ===")
                    print(result2.stdout)
        except FileNotFoundError:
            print("Slurm commands not available. Are you on a login/compute node?")

def on_logs(btn):
    monitor_output.clear_output()
    with monitor_output:
        # Find the most recent .out file
        pattern = os.path.join(PROJECT_DIR, f"{w_slurm_jobname.value}.*.out")
        files = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)

        if files:
            latest = files[0]
            print(f"=== Latest log: {latest} ===")
            print(f"(showing last 50 lines)\n")
            with open(latest) as f:
                lines = f.readlines()
                for line in lines[-50:]:
                    print(line, end="")
        else:
            print("No log files found yet. The job may not have started.")

        # Also check AutoGrow output directory
        output = w_output_folder.value
        if os.path.exists(output):
            gen_dirs = sorted(glob.glob(os.path.join(output, "generation_*")))
            if gen_dirs:
                print(f"\n=== AutoGrow4 Progress ===")
                print(f"Completed generations: {len(gen_dirs)}")
                for gd in gen_dirs[-3:]:
                    smi_files = glob.glob(os.path.join(gd, "*.smi"))
                    print(f"  {os.path.basename(gd)}: {len(smi_files)} .smi files")

def on_cancel(btn):
    monitor_output.clear_output()
    with monitor_output:
        if slurm_job_id[0]:
            result = subprocess.run(["scancel", slurm_job_id[0]], capture_output=True, text=True)
            print(f"Cancelled job {slurm_job_id[0]}")
            if result.stderr:
                print(f"Warning: {result.stderr}")
        else:
            print("No job ID tracked. Use: scancel <job_id>")

check_btn.on_click(on_check)
logs_btn.on_click(on_logs)
cancel_btn.on_click(on_cancel)

display(widgets.HBox([check_btn, logs_btn, cancel_btn]))
display(monitor_output)

---

## 7. Analyze Results

After the AutoGrow4 run completes, use the cells below to inspect the results.

In [ ]:
import glob
import os
import pandas as pd
from IPython.display import display, HTML

def load_ranked_smi(filepath):
    """Load a ranked .smi file into a DataFrame."""
    rows = []
    with open(filepath) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 3:
                rows.append({
                    "SMILES": parts[0],
                    "Name": parts[1],
                    "Fitness": float(parts[-2]) if len(parts) >= 3 else None,
                    "Diversity": float(parts[-1]) if len(parts) >= 4 else None,
                })
    return pd.DataFrame(rows)

# Find all ranked .smi files
output_dir = w_output_folder.value
ranked_files = sorted(glob.glob(os.path.join(output_dir, "generation_*", "*ranked*.smi")))

if ranked_files:
    display(HTML(f"<h4>Found {len(ranked_files)} generation result files</h4>"))

    # Show the last generation's top molecules
    latest = ranked_files[-1]
    display(HTML(f"<b>Latest: {os.path.basename(latest)}</b>"))

    df = load_ranked_smi(latest)
    if not df.empty:
        display(HTML(f"<p>Top 10 molecules (by fitness score):</p>"))
        display(df.head(10))
    else:
        print("No ranked molecules found in the file.")
else:
    display(HTML("<p>No results yet. Run an AutoGrow4 job first.</p>"))

In [ ]:
# ── Visualize fitness over generations ──
import matplotlib.pyplot as plt
import glob
import os
import numpy as np

output_dir = w_output_folder.value
ranked_files = sorted(glob.glob(os.path.join(output_dir, "generation_*", "*ranked*.smi")))

if ranked_files:
    gen_numbers = []
    best_scores = []
    avg_scores = []

    for rf in ranked_files:
        # Extract generation number from directory name
        gen_dir = os.path.basename(os.path.dirname(rf))
        gen_num = int(gen_dir.replace("generation_", ""))

        scores = []
        with open(rf) as f:
            for line in f:
                parts = line.strip().split("\t")
                if len(parts) >= 3:
                    try:
                        scores.append(float(parts[-2]))
                    except ValueError:
                        pass

        if scores:
            gen_numbers.append(gen_num)
            best_scores.append(min(scores))  # Lower = better for Vina
            avg_scores.append(np.mean(scores))

    if gen_numbers:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(gen_numbers, best_scores, 'o-', label="Best Score", color="#e74c3c", linewidth=2)
        ax.plot(gen_numbers, avg_scores, 's--', label="Average Score", color="#3498db", linewidth=1.5)
        ax.set_xlabel("Generation", fontsize=12)
        ax.set_ylabel("Docking Score (kcal/mol)", fontsize=12)
        ax.set_title("AutoGrow4 Fitness Progression", fontsize=14)
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("No results to plot yet.")

In [ ]:
# ── Visualize top molecule in NGL Viewer ──
import nglview as nv
import glob
import os

output_dir = w_output_folder.value

# Find docked PDBQT/PDB files from the latest generation
gen_dirs = sorted(glob.glob(os.path.join(output_dir, "generation_*")))

if gen_dirs:
    latest_gen = gen_dirs[-1]
    pdb_files = glob.glob(os.path.join(latest_gen, "**", "*.pdb"), recursive=True)
    pdbqt_files = glob.glob(os.path.join(latest_gen, "**", "*.pdbqt"), recursive=True)

    ligand_files = pdb_files or pdbqt_files

    if ligand_files:
        print(f"Found {len(ligand_files)} docked ligand files in {os.path.basename(latest_gen)}")
        print(f"Showing first ligand: {os.path.basename(ligand_files[0])}")

        # Load receptor + best ligand
        receptor_file = w_receptor.value
        if os.path.exists(receptor_file):
            view = nv.show_file(receptor_file)
            view.clear_representations()
            view.add_cartoon(selection="protein", color="sstruc", opacity=0.7)

            # Add ligand
            view.add_component(ligand_files[0])
            view[1].add_licorice(color="element")
            view[1].add_surface(opacity=0.3, color="element")

            display(view)
        else:
            print(f"Receptor file not found: {receptor_file}")
    else:
        print(f"No docked ligand files found in {os.path.basename(latest_gen)}")
else:
    print("No generation directories found. Run AutoGrow4 first.")

---

## 8. Quick Test Run (Local Validation)

Run a minimal AutoGrow4 test (1 generation, small population) to verify the installation works before submitting a full HPC job.

In [ ]:
import subprocess
import json
import os
from IPython.display import display, HTML
import ipywidgets as widgets

test_output = widgets.Output()
test_btn = widgets.Button(description="Run Quick Test", button_style="warning", layout=widgets.Layout(width="200px"))

def on_test(btn):
    test_output.clear_output()
    with test_output:
        print("Starting quick validation test...")
        print("(1 generation, 3 mutants, 3 crossovers — should complete in a few minutes)\n")

        test_config = {
            "filename_of_receptor": os.path.join(RECEPTOR_DIR, "2ZSH.pdbqt"),
            "center_x": 51.0,
            "center_y": 59.5,
            "center_z": 37.4,
            "size_x": 25.0,
            "size_y": 16.0,
            "size_z": 25.0,
            "source_compound_file": os.path.join(AUTOGROW_DIR, "source_compounds", "naphthalene_smiles.smi"),
            "root_output_folder": os.path.join(PROJECT_DIR, "test_output"),
            "number_of_mutants_first_generation": 3,
            "number_of_crossovers_first_generation": 3,
            "number_of_mutants": 3,
            "number_of_crossovers": 3,
            "number_elitism_advance_from_previous_gen": 3,
            "top_mols_to_seed_next_generation": 3,
            "diversity_mols_to_seed_first_generation": 1,
            "diversity_seed_depreciation_per_gen": 0,
            "num_generations": 1,
            "number_of_processors": 1,
            "multithread_mode": "serial",
            "selector_choice": "Rank_Selector",
            "max_variants_per_compound": 1,
            "filter_source_compounds": False,
            "use_docked_source_compounds": False,
            "LipinskiLenientFilter": True,
            "docking_exhaustiveness": 1,
            "start_a_new_run": True,
            "scoring_choice": "VINA",
            "dock_choice": "QuickVina2Docking",
            "gypsum_timeout_limit": 1,
            "rxn_library": "all_rxns",
        }

        # Add conversion tool
        mgl_path = os.path.join(MGLTOOLS_DIR, "bin", "pythonsh")
        obabel_path = os.popen("which obabel 2>/dev/null").read().strip()

        if os.path.exists(mgl_path):
            test_config["mgltools_directory"] = MGLTOOLS_DIR
            test_config["conversion_choice"] = "MGLToolsConversion"
        elif obabel_path:
            test_config["obabel_path"] = obabel_path
            test_config["conversion_choice"] = "ObabelConversion"
        else:
            print("WARNING: Neither MGLTools nor obabel found!")
            print("Install one of them before running.")
            return

        # Save test config
        test_json = os.path.join(PROJECT_DIR, "test_config.json")
        os.makedirs(test_config["root_output_folder"], exist_ok=True)
        with open(test_json, "w") as f:
            json.dump(test_config, f, indent=2)

        print(f"Config saved to: {test_json}")
        print(f"Output will go to: {test_config['root_output_folder']}")
        print("\nRunning AutoGrow4...\n" + "=" * 60)

        try:
            proc = subprocess.run(
                ["python", "run_autogrow.py", "-j", test_json],
                cwd=AUTOGROW_DIR,
                capture_output=True, text=True,
                timeout=600  # 10 minute timeout
            )

            # Show output
            if proc.stdout:
                for line in proc.stdout.split("\n")[-30:]:
                    print(line)

            if proc.returncode == 0:
                display(HTML("<p style='color:green; font-size:16px'><b>Test completed successfully!</b></p>"))

                # Show results
                import glob
                results = glob.glob(os.path.join(test_config["root_output_folder"], "**", "*ranked*.smi"), recursive=True)
                if results:
                    print(f"\nResult files generated: {len(results)}")
                    for r in results:
                        print(f"  - {r}")
                        with open(r) as rf:
                            lines = rf.readlines()
                            print(f"    ({len(lines)} molecules ranked)")
            else:
                display(HTML("<p style='color:red'><b>Test failed.</b></p>"))
                if proc.stderr:
                    print("\nSTDERR:")
                    for line in proc.stderr.split("\n")[-20:]:
                        print(line)

        except subprocess.TimeoutExpired:
            display(HTML("<span style='color:orange'><b>Test timed out (10 min). This may be normal for slow systems.</b></span>"))
        except Exception as e:
            display(HTML(f"<span style='color:red'>Error: {e}</span>"))

test_btn.on_click(on_test)

display(test_btn)
display(test_output)

---

## 9. Save Results to Project Directory

Copy the final results (ranked molecules, plots, configs) to a clean project output folder.

In [ ]:
import shutil
import glob
import os
from datetime import datetime

results_dir = os.path.join(PROJECT_DIR, f"results_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
os.makedirs(results_dir, exist_ok=True)

output_dir = w_output_folder.value

# Copy ranked .smi files
smi_files = glob.glob(os.path.join(output_dir, "**", "*ranked*.smi"), recursive=True)
for f in smi_files:
    shutil.copy2(f, results_dir)

# Copy any plots
plot_files = glob.glob(os.path.join(output_dir, "**", "*.png"), recursive=True)
for f in plot_files:
    shutil.copy2(f, results_dir)

# Copy config and slurm script
for fname in ["autogrow_config.json", "autogrow_slurm.sh"]:
    src = os.path.join(PROJECT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, results_dir)

# Copy log files
log_files = glob.glob(os.path.join(PROJECT_DIR, f"{w_slurm_jobname.value}.*.out"))
for f in log_files:
    shutil.copy2(f, results_dir)

print(f"Results saved to: {results_dir}")
print(f"\nContents:")
for f in sorted(os.listdir(results_dir)):
    size = os.path.getsize(os.path.join(results_dir, f))
    print(f"  {f} ({size:,} bytes)")

print(f"\nDone! Your AutoGrow4 run results are saved in:\n  {results_dir}")